> **Research record.** Cleaned copy of `XXX-calculate-WFH/101-remote-work-statistical-areas.ipynb` (data checks, dead ends and unlabelled plots removed; outputs cleared).
> Spatial diagnostics of the same threshold when the metric is aggregated to statistical areas.
>
> Requires the authors' BigQuery project and the full GPS dataset, which are not public; the notebook is kept as a record of the analysis behind the paper. Run it with its own folder as the working directory and `settings.ini` at the repository root. For a runnable walk-through on synthetic data see `demo/`.

# Statistical-area threshold diagnostics

This cleaned notebook keeps only the spatial diagnostics needed to justify filtering user-month records with `total_work_days >= 7` before aggregating WFH to statistical areas.

In [ ]:
import os
import sys

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

project_root = "../.."
if project_root not in sys.path:
    sys.path.append(project_root)

from geofunctions import utils

MIN_TOTAL_WORK_DAYS = 7
REMOTE_WORK_FEATURE = "flag_remote_work_smoothed_max_thresh"
sns.set_theme(style="whitegrid")

In [ ]:
months = utils.return_list_months_ok()
list_months = []

for month in months:
    try:
        df_part = pd.read_csv(
            utils.get_path("processed", "dwells", f"remote_work/user_remote_work_{month}.csv")
        )
        df_part["month"] = month
        list_months.append(df_part)
    except FileNotFoundError:
        print(f"Missing user_remote_work file for {month}")

df_shares_all = pd.concat(list_months, ignore_index=True)
df_shares = df_shares_all[df_shares_all["total_work_days"] >= MIN_TOTAL_WORK_DAYS].copy()

print(f"User-months before threshold: {len(df_shares_all):,}")
print(f"User-months after threshold:  {len(df_shares):,}")
print(f"Retained share: {len(df_shares) / len(df_shares_all):.1%}")

## Monthly user retention

First check whether the `>= 7` threshold keeps a usable number of users in every month.

In [ ]:
monthly_users = pd.concat(
    [
        df_shares_all.groupby("month")["identifier"].nunique().rename("all_users"),
        df_shares.groupby("month")["identifier"].nunique().rename("kept_users"),
    ],
    axis=1,
).reset_index()
monthly_users["kept_share"] = monthly_users["kept_users"] / monthly_users["all_users"]

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
sns.lineplot(data=monthly_users, x="month", y="kept_users", marker="o", ax=axes[0])
sns.lineplot(data=monthly_users, x="month", y="kept_share", marker="o", ax=axes[1])
axes[0].set_ylabel("Users retained")
axes[1].set_ylabel("Share retained")
axes[1].set_ylim(0, 1)
axes[1].tick_params(axis="x", rotation=90)
fig.suptitle(f"User retention after total_work_days >= {MIN_TOTAL_WORK_DAYS}", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
home_rows = []
for month in sorted(df_shares["month"].unique()):
    try:
        home_df = pd.read_csv(
            os.path.join(
                utils.get_path("processed", "dwells", "home"),
                f"home_geohashes_night_shabbat_{month}.csv",
            )
        )
        home_df["month"] = month
        home_rows.append(home_df[["identifier", "geohash", "month"]])
    except FileNotFoundError:
        print(f"Missing home file for {month}")

home_df_full = pd.concat(home_rows, ignore_index=True)
geohash_statarea = gpd.read_file(
    utils.get_path("processed", "adm", "geohash_stat_area_pairs.geojson")
)

df_merge = (
    df_shares[["identifier", REMOTE_WORK_FEATURE, "month"]]
    .merge(home_df_full, on=["identifier", "month"], how="inner")
    .merge(geohash_statarea[["geohash", "SHEM_YISHUV_HEB", "YISHUV_STAT_2022"]], on="geohash", how="inner")
)

df_wfh_stat_areas = (
    df_merge.groupby(["SHEM_YISHUV_HEB", "YISHUV_STAT_2022", "month"])[REMOTE_WORK_FEATURE]
    .agg(num_homes="count", WFH_mean="mean")
    .reset_index()
)

## Statistical-area sample size after threshold

After filtering weak user-months, each statistical-area/month WFH estimate should still be based on enough homes.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.histplot(df_wfh_stat_areas["num_homes"], bins=60, ax=ax)
ax.set_xlim(0, df_wfh_stat_areas["num_homes"].quantile(0.99))
ax.set_title(f"Homes per statistical-area/month after total_work_days >= {MIN_TOTAL_WORK_DAYS}")
ax.set_xlabel("num_homes")
ax.set_ylabel("Statistical-area/months")
plt.show()

## Spatial coverage over time

This plot checks whether the threshold leaves broad spatial coverage by month.

In [ ]:
coverage = (
    df_wfh_stat_areas.groupby("month")
    .agg(
        stat_areas_any=("YISHUV_STAT_2022", "nunique"),
        stat_area_months_min3=("num_homes", lambda s: (s >= 3).sum()),
        median_num_homes=("num_homes", "median"),
    )
    .reset_index()
)

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
sns.lineplot(data=coverage, x="month", y="stat_areas_any", marker="o", label="any homes", ax=axes[0])
sns.lineplot(data=coverage, x="month", y="stat_area_months_min3", marker="o", label=">= 3 homes", ax=axes[0])
sns.lineplot(data=coverage, x="month", y="median_num_homes", marker="o", color="purple", ax=axes[1])
axes[0].set_ylabel("Statistical areas")
axes[1].set_ylabel("Median num_homes")
axes[1].tick_params(axis="x", rotation=90)
fig.suptitle(f"Spatial coverage after total_work_days >= {MIN_TOTAL_WORK_DAYS}", y=1.02)
plt.tight_layout()
plt.show()

## WFH distribution after threshold

The threshold should not create pathological WFH values; this shows the resulting distribution of statistical-area means.

In [ ]:
plt.figure(figsize=(9, 5))
sns.histplot(df_wfh_stat_areas["WFH_mean"].dropna(), bins=40)
plt.title(f"Distribution of statistical-area WFH means after total_work_days >= {MIN_TOTAL_WORK_DAYS}")
plt.xlabel("WFH_mean")
plt.ylabel("Statistical-area/months")
plt.tight_layout()
plt.show()